# 03. Проверяем ответы через Ragas

**Цель:** различить ошибки поиска и генерации на сохранённых ответах.
Индекс и генератор больше не нужны: берём точные контексты из
[02](02_hybrid_rag_inference.ipynb). Нужны судья и embeddings для Answer Relevancy.
[Настройка](SETUP.md).

```text
question + response + retrieved_contexts + reference
                           ↓
                метрика Ragas → оценка + статус
```

**Run All вызывает модель-судью:** сначала на одном примере, затем на трёх вопросах.
Эти учебные одиночные вызовы не записываются в основной отчёт и при повторении
считаются снова. Пакетный расчёт сохраняется и возобновляется.
Метрики — измерения качества, а не гарантия истинности ответа или обычные unit-тесты.

## 1. Какие данные проверяет судья

`question` — исходный вопрос; `response` — сохранённый ответ модели;
`retrieved_contexts` — блоки, фактически переданные генератору;
`reference` — независимо подготовленный эталон по статье.
Эталон не был доступен генератору. Вопросы без эталона проверяют отсутствие
данных или неверные предпосылки и требуют также ручного разбора.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "helpers" / "artifacts.py").exists():
    ROOT = ROOT / "hybrid_rag_notebooks"
if not (ROOT / "helpers" / "artifacts.py").exists():
    raise RuntimeError("Запустите Jupyter из hybrid_rag_notebooks или её родительской папки")
sys.path.insert(0, str(ROOT))
from helpers.settings import prepare_environment  # noqa: E402

prepare_environment(ROOT)

In [ ]:
import json
import math
import os

import pandas as pd
from helpers.artifacts import append_jsonl, ensure_manifest, fingerprint, latest_records
from helpers.evaluation import (
    CONTEXT_METRICS,
    METRIC_NAMES,
    REFERENCE_METRICS,
    _authentication_error,
    score_summary,
)
from helpers.inspection import filter_report, print_example
from helpers.settings import judge_settings
from tqdm.auto import tqdm

RUN_NAME = "attention_multimodal_hybrid_v1"
EVAL_NAME = "ragas_multimodal_v1"
RUN_DIR = ROOT / "outputs" / "runs" / RUN_NAME
EVAL_DIR = RUN_DIR / "evaluations" / EVAL_NAME
JUDGE = judge_settings()
# Для другого судьи выберите новый EVAL_NAME.

In [ ]:
def load_run(run_dir):
    path = run_dir / "run.json"
    if not path.exists():
        raise FileNotFoundError(f"Нет {path}; сохраните пакет в ноутбуке 02")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest["fingerprint"] != fingerprint(manifest["config"]):
        raise ValueError("Повреждён run.json")
    questions = manifest["config"]["questions"]
    answers = latest_records(run_dir / "answers.jsonl")
    for sample in questions:
        row = answers.get(sample["id"])
        if row and row.get("status") == "ok":
            if row.get("question") != sample["question"] or row.get("reference") != sample.get(
                "reference"
            ):
                raise ValueError(f"Ответ не соответствует набору: {sample['id']}")
            if not isinstance(row.get("response"), str) or not row["response"].strip():
                raise ValueError(f"Пустой ответ: {sample['id']}")
            contexts = row.get("retrieved_contexts")
            if not isinstance(contexts, list) or not all(isinstance(c, str) for c in contexts):
                raise ValueError("Неверный формат контекстов")
            if row.get("context") != "\n\n".join(contexts):
                raise ValueError("Сохранённые контексты отличаются от контекста промпта")
    return manifest, questions, answers

In [ ]:
run_manifest, questions, answers = load_run(RUN_DIR)
print(
    pd.DataFrame(
        [
            {
                "id": q["id"],
                "reference": bool(q.get("reference")),
                "status": answers.get(q["id"], {}).get("status", "missing"),
            }
            for q in questions
        ]
    ).to_string(index=False)
)
sample = next(
    (
        answers[q["id"]]
        for q in questions
        if answers.get(q["id"], {}).get("status") == "ok"
        and answers[q["id"]].get("retrieved_contexts")
    ),
    None,
)
if sample is None:
    raise ValueError(
        "Для учебного примера нужен хотя бы один успешный ответ с контекстом из ноутбука 02"
    )
print(
    json.dumps(
        {k: sample.get(k) for k in ("question", "response", "retrieved_contexts", "reference")},
        ensure_ascii=False,
        indent=2,
    )
)

## 2. Пять метрик и подключения

| Метрика | На какой вопрос отвечает | Необходимые данные |
|---|---|---|
| Faithfulness | Подтверждаются ли утверждения ответа источниками? | вопрос, ответ, контексты |
| Answer Relevancy | Отвечает ли ответ на вопрос? | вопрос, ответ; судья и embeddings |
| Context Precision with Reference | Полезны ли первые контексты для эталона? | вопрос, контексты, эталон |
| Context Recall | Покрывают ли контексты факты эталона? | вопрос, контексты, эталон |
| Factual Correctness | Насколько совпадают факты ответа и эталона? | ответ, эталон; здесь F1 |

Faithfulness может быть высокой у неполного ответа. Relevancy не доказывает
фактическую правильность. Ошибка в эталоне влияет на метрики с эталоном.
Оценки зависят от модели-судьи, его промптов и языка; универсального порога нет.
Используем API `ragas.metrics.collections` из закреплённой в requirements версии.

In [ ]:
from openai import AsyncOpenAI
from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory

judge_client = AsyncOpenAI(
    base_url=JUDGE["base_url"],
    api_key=os.getenv("JUDGE_API_KEY") or os.getenv("LLM_API_KEY") or "local-no-key",
    timeout=JUDGE["timeout"],
    max_retries=2,
)
judge_embed_client = AsyncOpenAI(
    base_url=JUDGE["embedding_url"],
    api_key=os.getenv("JUDGE_EMBED_API_KEY") or os.getenv("EMBED_API_KEY") or "local-no-key",
    timeout=JUDGE["timeout"],
    max_retries=2,
)
llm = llm_factory(JUDGE["model"], client=judge_client, temperature=JUDGE["temperature"])
embeddings = embedding_factory("openai", model=JUDGE["embedding_model"], client=judge_embed_client)

In [ ]:
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    FactualCorrectness,
    Faithfulness,
)

metrics = {
    "faithfulness": Faithfulness(llm=llm),
    "answer_relevancy": AnswerRelevancy(
        llm=llm, embeddings=embeddings, strictness=JUDGE["strictness"]
    ),
    "context_precision_with_reference": ContextPrecisionWithReference(llm=llm),
    "context_recall": ContextRecall(llm=llm),
    "factual_correctness": FactualCorrectness(llm=llm, mode="f1"),
}

## 3. Одна метрика, один ответ

Это прямой вызов Ragas. Печатаем возвращённый объект и числовую оценку;
состав дополнительных полей зависит от метрики и установленной версии.
Один `ascore` может вызвать судью несколько раз, например для выделения и проверки утверждений.

In [ ]:
single_result = await metrics["faithfulness"].ascore(
    user_input=sample["question"],
    response=sample["response"],
    retrieved_contexts=sample["retrieved_contexts"],
)
print("Результат Ragas:", single_result)
print("Faithfulness:", single_result.value)

### Управляемая ошибка

Добавим к ответу заведомо постороннее утверждение. Вопрос и контекст остаются
прежними. Проверяем чувствительность судьи к изменению ответа; снижение оценки
не гарантируется, особенно если ответ уже содержал ошибки. Это отдельный
учебный эксперимент, исходный ответ на диске не меняется.

In [ ]:
changed_response = sample["response"] + " В статье также утверждается, что Луна состоит из сыра."
changed_result = await metrics["faithfulness"].ascore(
    user_input=sample["question"],
    response=changed_response,
    retrieved_contexts=sample["retrieved_contexts"],
)
print("Исходный ответ:", sample["response"])
print("Изменённый ответ:", changed_response)
print("Было:", single_result.value, "стало:", changed_result.value)

## 4. Пакетный расчёт с сохранением

Раскрываем маршрутизацию полей по метрикам. Пропуск не равен нулю:
без эталона три метрики неприменимы; без контекста пропускаем контекстные;
неудачный inference не оцениваем. Неопределённое число получает `undefined`,
ошибка API — `error`. При 401/403 останавливаемся и сохраняем прогресс.

In [ ]:
async def score_one(row, name):
    contexts = row.get("retrieved_contexts", [])
    if row.get("status") != "ok":
        return {
            "status": "skipped",
            "reason": "inference_failed_or_missing",
            "value": None,
        }
    if name in REFERENCE_METRICS and not row.get("reference"):
        return {"status": "skipped", "reason": "no_reference", "value": None}
    if name in CONTEXT_METRICS and not contexts:
        return {"status": "skipped", "reason": "empty_context", "value": None}
    try:
        metric = metrics[name]
        if name == "faithfulness":
            result = await metric.ascore(
                user_input=row["question"],
                response=row["response"],
                retrieved_contexts=contexts,
            )
        elif name == "answer_relevancy":
            result = await metric.ascore(user_input=row["question"], response=row["response"])
        elif name in ("context_precision_with_reference", "context_recall"):
            result = await metric.ascore(
                user_input=row["question"],
                reference=row["reference"],
                retrieved_contexts=contexts,
            )
        elif name == "factual_correctness":
            result = await metric.ascore(response=row["response"], reference=row["reference"])
        else:
            raise ValueError(f"Неизвестная метрика: {name}")
        value = float(result.value)
        if not math.isfinite(value):
            return {"status": "undefined", "reason": "nonfinite_metric", "value": None}
        return {"status": "ok", "value": value}
    except Exception as exc:
        failure = {
            "status": "error",
            "error": f"{type(exc).__name__}: {exc}",
            "value": None,
        }
        if _authentication_error(exc):
            failure["reason"] = "authentication_error"
        return failure

In [ ]:
def report():
    manifest, questions, answers = load_run(RUN_DIR)
    judge_path = EVAL_DIR / "judge.json"
    if judge_path.exists():
        bound = json.loads(judge_path.read_text(encoding="utf-8"))["config"]
        if bound["run_fingerprint"] != manifest["fingerprint"]:
            raise ValueError("Оценка относится к другому запуску")
    scores = latest_records(EVAL_DIR / "scores.jsonl", ("id", "metric"))
    # Сводка по всему плану. При изменении ответа старые оценки становятся missing.
    details, summary = score_summary(questions, answers, scores, METRIC_NAMES)
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    details.to_csv(EVAL_DIR / "details.csv", index=False)
    summary.to_csv(EVAL_DIR / "summary.csv", index=False)
    return details, summary

In [ ]:
METRICS = list(METRIC_NAMES)
LIMIT = 3  # None — все вопросы. После изменения повторите расчёт ниже.
print("Метрики:", METRICS, "вопросов:", LIMIT or "все")

In [ ]:
import ragas

if not METRICS or set(METRICS) - set(METRIC_NAMES):
    raise ValueError("Выберите известные метрики")
if LIMIT is not None and LIMIT < 1:
    raise ValueError("LIMIT должен быть положительным или None")
run_manifest, questions, answers = load_run(RUN_DIR)
eval_config = {
    "run_fingerprint": run_manifest["fingerprint"],
    "judge": JUDGE,
    "ragas_version": ragas.__version__,
    "metrics": list(METRIC_NAMES),
}
ensure_manifest(EVAL_DIR / "judge.json", eval_config)
previous = latest_records(EVAL_DIR / "scores.jsonl", ("id", "metric"))
work = []
for q in questions[:LIMIT]:
    row = answers.get(q["id"], {**q, "status": "missing"})
    for name in METRICS:
        old = previous.get((q["id"], name), {})
        if old.get("answer_fingerprint") == fingerprint(row) and old.get("status") in (
            "ok",
            "skipped",
        ):
            continue
        work.append((q["id"], row, name))
print(
    f"Полный план: {len(questions) * len(METRIC_NAMES)}; выбрано: {len(questions[:LIMIT]) * len(METRICS)}; осталось: {len(work)}",
    flush=True,
)
for sample_id, row, name in tqdm(work, desc="Ragas", unit="метрика"):
    result = await score_one(row, name)
    append_jsonl(
        EVAL_DIR / "scores.jsonl",
        {
            "id": sample_id,
            "metric": name,
            "answer_fingerprint": fingerprint(row),
            **result,
        },
    )
    if result["status"] in ("error", "undefined"):
        tqdm.write(f"{sample_id} / {name}: {result}")
    if result.get("reason") == "authentication_error":
        report()
        raise RuntimeError(
            "API судьи вернул 401/403; проверьте JUDGE_API_KEY и JUDGE_EMBED_API_KEY. Прогресс сохранён"
        )
details, summary = report()
print(summary.to_string(index=False))

## 5. Разбор ошибок, а не только среднее

`planned` — весь набор, `ok` — конечные оценки, `skipped` — неприменимые,
`error` — ошибки вызовов, `undefined` — неопределённые значения,
`missing` — ещё не рассчитанные или устаревшие оценки. Среднее учитывает только `ok`.
После пробы на трёх вопросах много `missing` ожидаемо: для полного расчёта задайте `LIMIT=None`.
Просмотр ниже не вызывает судью и доступен отдельно от расчёта.

In [ ]:
details, summary = report()
print(summary.to_string(index=False))
print("CSV:", EVAL_DIR)

In [ ]:
METRIC = "faithfulness"
STATUS = "all"  # all, problems, ok
GROUP = "all"  # all, reference, no_reference
examples = filter_report(details, questions, answers, metric=METRIC, status=STATUS, group=GROUP)
print(examples.head(10).to_string(index=False))
SAMPLE_ID = ""  # Пустое значение выбирает первый пример.
sample_id = SAMPLE_ID or (examples.iloc[0]["id"] if len(examples) else None)
if sample_id:
    print_example(sample_id, details, answers, questions)

Для каждого слабого ответа пройдите цепочку:

1. **Нужного факта нет в контексте:** проверьте извлечение PDF, чанки, выдачи и top-k в 01/02.
2. **Факт есть, ответ его исказил:** проверьте промпт и генерацию в 02.
3. **Метрика не согласуется с ручной проверкой:** проверьте эталон и вывод судьи.

Это гипотезы для диагностики, а не автоматическая классификация причин по баллам.
Вопрос `attn_30_french_discrepancy` требует проверить оба значения BLEU в статье.
Для `GROUP="no_reference"` вручную проверьте отказ: корректный отказ может получить
низкую Relevancy или неопределённую Faithfulness. Ожидаемое поведение выводится рядом с ответом.

## 6. Сравнение двух запусков

Во втором ноутбуке сохраните другой режим поиска под новым `RUN_NAME`, затем
оцените его с тем же судьёй и набором метрик. Ниже укажите имя этого запуска.
Сравниваем **одни и те же успешно оценённые пары вопрос–метрика** и показываем их число.
Несовпадающие вопросы, эталоны, версия Ragas или настройки судьи вызывают ошибку.
Разница средних на небольшом наборе не является проверкой статистической значимости.

In [ ]:
OTHER_RUN_NAME = ""  # Например, attention_multimodal_dense_v1; пусто — пропустить.
OTHER_EVAL_NAME = EVAL_NAME
if OTHER_RUN_NAME:
    other_dir = ROOT / "outputs" / "runs" / OTHER_RUN_NAME
    other_manifest, other_questions, other_answers = load_run(other_dir)
    if other_questions != questions:
        raise ValueError("Для сравнения нужны одинаковые вопросы и эталоны")
    other_eval = other_dir / "evaluations" / OTHER_EVAL_NAME
    other_config = json.loads((other_eval / "judge.json").read_text(encoding="utf-8"))["config"]
    this_config = json.loads((EVAL_DIR / "judge.json").read_text(encoding="utf-8"))["config"]
    if other_config["run_fingerprint"] != other_manifest["fingerprint"]:
        raise ValueError("Второй отчёт относится к другому запуску")
    if {k: v for k, v in this_config.items() if k != "run_fingerprint"} != {
        k: v for k, v in other_config.items() if k != "run_fingerprint"
    }:
        raise ValueError("Судья, версия Ragas и метрики должны совпадать")
    other_scores = latest_records(other_eval / "scores.jsonl", ("id", "metric"))
    other_details, _ = score_summary(other_questions, other_answers, other_scores, METRIC_NAMES)
    current_details, _ = report()  # Перечитываем актуальные ответы и оценки.
    paired = current_details.merge(
        other_details, on=["id", "metric"], suffixes=("_current", "_other")
    )
    valid = paired[(paired.status_current == "ok") & (paired.status_other == "ok")].copy()
    valid["delta"] = valid.value_current - valid.value_other
    comparison = valid.groupby("metric").agg(
        n=("id", "size"),
        current=("value_current", "mean"),
        other=("value_other", "mean"),
        delta=("delta", "mean"),
    )
    print(comparison.reindex(METRIC_NAMES).to_string())
    print("Положительная delta: среднее текущего запуска выше. NaN: нет общих успешных пар.")
else:
    print("Сначала оцените второй запуск; затем задайте OTHER_RUN_NAME")

Меняйте один параметр за эксперимент. Сравните полный набор вопросов и изучите
отдельные ответы, прежде чем выбирать настройки. Исходные артефакты остаются на диске;
после остановки повторите пакетную ячейку, а после смены судьи используйте новый `EVAL_NAME`.

In [ ]:
for name in ("judge_client", "judge_embed_client"):
    client = globals().get(name)
    if client is not None:
        await client.close()